# ===========================================
### repex_topology_parser
#### Currently only supports Amber Potential 
#### without CMAP corrections
$$ E_{\rm total} = \sum_{\rm bonds} K_r (r - r_{eq})^2 
                     + \sum_{\rm angles} K_\theta (\theta - \theta_{eq})^2
                     + \sum_{\rm dihedrals} {V_n \over 2} 
                                       [1 + {\rm cos}(n\phi - \gamma)]\\ 
                     + \sum_{i<j} \epsilon_{ij} \left [ {\left(\frac{\sigma_{ij}}{R_{ij}}\right)}^{12} - 
                                          {\left(\frac{\sigma_{ij}}{R_{ij}}\right)}^6 \right]
                     + \sum_{i<j} {q_iq_j \over \epsilon R_{ij}} 
                                 
                                 $$
### TODO
#### - Add CMAP lambda scaling
#### - Extend to CHARMM, OPLS-AA
### Examples for REST2, and ssREST2 (solvent-scaled REST2)
# ===========================================

# ===========================================
### Energy Components
$$ E_{\rm total} = E_{\rm protein-protein} + E_{\rm protein-water} + E_{\rm water-water} $$
#### REST2 scaling with $\lambda$
$$ \lambda = \frac{\beta_n}{\beta_0} = \frac{T_0}{T_n} : \beta_n = \frac{1}{k_B T_n} $$
$$ E_{\rm total}^{\lambda_n} = \lambda_n E_{\rm protein-protein} + \sqrt{\lambda_n} E_{\rm protein-water} + E_{\rm water-water} $$
### solvent-scaling 
$$ \kappa_i =  e^{\frac{i}{N - 1} log(\kappa_{max})} ; i \in [0,N-1]$$
$$ E_{\rm total}^{\kappa_n} = E_{\rm protein-protein} + \kappa_n E_{\rm protein-water} + E_{\rm water-water} $$
### ssREST2 (solvent-scaled REST2)
$$ E_{\rm total}^{\lambda_n,\kappa_n} = \lambda_nE_{\rm protein-protein} + \boldsymbol{\kappa_n} \sqrt{\lambda_n} E_{\rm protein-water} + E_{\rm water-water} $$

# ===========================================

In [2]:
# Load our library
import src.repex_topology_parser as rtp

In [3]:
# initialize our class providing an input processed.top file
test_module = rtp.topo2rest('./tests/topology_files/test_topo/processed.top')

In [4]:
### Let us display our molecules contained in our topology
test_module.molecules

{0: 'Protein_chain_A', 1: 'HxD', 2: 'SOL', 3: 'NA', 4: 'CL'}

### Here we perform with one command rest2 scaling. The hot molecule will have dihedrals/charges/LJ 
### parameters scaled by $\sqrt\lambda$. Thus hot molecule-hot molecule interactions will be scaled
### by $\lambda$, while hot molecule - other molecules will be scaled by $\sqrt\lambda$

In [5]:
# Our first example is performing solute scaling (REST2) on just the protein
REST2 = { 'hot_molecules':[0], # Select the molecule(s) you desire to scale, as a list
            'nreps':20, # define the number of replicas you desire
            'outfile':'topol_rest2', # provide a prefix name for your topology
                                     # default = 'topol'
            'filepath':'./test_run/', # define the directory you wish to write your scaled topologies
                                      # default='./'
            'method':'rest2', # method is rest2
            'temps':[300,500], # temperature range 300 to 500, utilized to compute the geometric
                               # temperature ladder
                               # default = [300,500]
            'verbose':True     # be verbose, important if you are performing scaling interactively
                               # e.g. not providing one or more of these options
            }
test_module.run(**REST2)

Running rest2 scaling method


### Here we perform with one command rest2 scaling with an additional scaling on the OW atom of our water model 
### In this case \'OW_tip4pd\' atomtype. First, the input of a hot molecule has the same effect as rest2 where the 
### protein dihedrals/charges/LJ parameters are scaled by $\lambda$, second the LJ $\epsilon$ of water is scaled 
### by $\kappa^2$ tuning the solvation of the hot molecule(s), and lastly, all non-hot molecule-water LJ 
### parameters are reset thus avoiding the effects of solvent scaling. 

In [6]:
test_module._get_molecule_atomtypes(2)

array(['HW', 'MW', 'OW_tip4pd'], dtype=object)

In [7]:
# Our second example is performing solvent-scaling REST3 (ssREST3) on just the protein
# From the displayed atomtypes contained in our solvent molecule we opt to scale
# 'OW_tip4pd' ('HW' and 'MW' have epsilon = 0.0 so we exclude these from the list)
ssREST3 = { 'hot_molecules':[0], # Select the molecule(s) you desire to scale, as a list
            'nreps':20, # define the number of replicas you desire
            'outfile':'topol_ssrest3', # provide a prefix name for your topology
                                     # default = 'topol'
            'filepath':'./test_run/', # define the directory you wish to write your scaled topologies
                                      # default='./'
            'method':'ssrest3', # method is ssrest3
            'temps':[300,500], # temperature range 300 to 500, utilized to compute the geometric
                               # temperature ladder
                               # default = [300,500]
            'kappa_low_temp' : 300, # At which temperature to activate solvent scaling, maybe useful
                                    # to increse to 330 when using a lower number of replicas so the base replica
                                    # experiences solvation more accurately. For ssREST3 simulations with 
                                    # 16 or more replicas, it is unlikedly changing this value to 330 will have 
                                    # any benefit. 
            'kappa_max' : 1.1, # Maximum kappa value
            'kappa_atom_names' : ['OW_tip4pd'], # List of atomtypes to apply kappa scaling
            'verbose':True     # be verbose, important if you are performing scaling interactively
                               # e.g. not providing one or more of these options
            }
test_module.run(**ssREST3)

Running ssrest3 scaling method
